In [1]:
import yaml
import os
import shutil
import copy
import subprocess

In [2]:
bar_1_template = {
    'type': 'single',
    'input mesh file': 'bar-1.g',
    'output mesh file': 'bar-1.e',
    'model': {
        'type': 'solid mechanics',
        'material': {
            'blocks': {
                'fine': 'elastic'
            },
            'elastic': {
                'model': 'linear elastic',
                'elastic modulus': 2.0684e11,
                "Poisson's ratio": 0.25,
                'density': 7.8441e3
            }
        }
    },
    'time integrator': None,
    'initial conditions': {
        'displacement': [
            {
                'node set': 'nsall',
                'component': 'x',
                'function': '{replace}' #str("\"5.1359 * t\"")
            }
        ]
    },
    'boundary conditions': {
        'Schwarz contact': [
            {
                'side set': 'ssx+',
                'source': 'bar-2.yaml',
                'source block': 'coarse',
                'source side set': 'ssx-',
                'friction type': 'frictionless',
                'swap BC types': 'false'
            }
        ]
    },
    'solver': None
}

bar_2_template = {
    'type': 'single',
    'input mesh file': 'bar-2.g',
    'output mesh file': 'bar-2.e',
    'model': {
        'type': 'solid mechanics',
        'material': {
            'blocks': {
                'coarse': 'elastic'
            },
            'elastic': {
                'model': 'linear elastic',
                'elastic modulus': 2.0684e11,
                "Poisson's ratio": 0.25,
                'density': 7.8441e3
            }
        }
    },
    'time integrator': None,
    'initial conditions': {
        'displacement': [
            {
                'node set': 'nsall',
                'component': 'x',
                'function': '{-replace}'
            }
        ]
    },
    'boundary conditions': {
        'Schwarz contact': [
            {
                'side set': 'ssx-',
                'source': 'bar-1.yaml',
                'source block': 'fine',
                'source side set': 'ssx+',
                'friction type': 'frictionless',
                'swap BC types': 'false'
            }
        ]
    },
    'solver': None
}

solver_templates = [
    {
        'type': 'Hessian minimizer',
        'step': 'full Newton',
        'minimum iterations': 1,
        'maximum iterations': 16,
        'relative tolerance': 1.0e-07,
        'absolute tolerance': 1.0e-03
    },
    {
        'type': 'explicit solver',
        'step': 'explicit'
    }
]

integrator_templates = [
    {
        'type': 'Newmark',
        'β': 0.25,
        'γ': 0.5
    },
    {
        'type': 'central difference',
        'CFL': 0.1,
        'γ': 0.5
    }
]

schwarz_main_template = {
    'type': 'multi',
    'domains': ['bar-1.yaml', 'bar-2.yaml'],
    'Exodus output interval': 1,
    'CSV output interval': None,
    'initial time': 0,
    'time step': None,
    'final time': 40e-5,
    'relaxation parameter': 0.8,
    'naive stabilized': True,
    'minimum iterations': 1,
    'maximum iterations': 16,
    'relative tolerance': 1.0e-07,
    'absolute tolerance': 1.0e-03
}


In [3]:
def replace_placeholders_in_file(filename):
    # Read the content of the file
    with open(filename, 'r', encoding='utf-8') as file:
        content = file.read()

    # Replace placeholders with the desired strings
    content = content.replace("'{replace}'", '\"5.1359 * t\"')
    content = content.replace("'{-replace}'", '\"-5.1359 * t\"')

    # Write the modified content back to the file
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(content)

    print(f"Replacements made in '{filename}' successfully.")

In [ ]:
case_map = {
    0: [ 0, 0, False ],
    1: [ 0, 1, False ],
    2: [ 0, 1, True ],
    3: [ 1, 0, False ]
}
main_yaml_ts = [ 1.0e-8, 1.0e-8, 1.0e-8, 1.0e-8 ]
def create_cases(mesh_folder, start_case, ts, csv_output_int):
    start_cases = range(start_case, start_case + 4)
    pwd = os.getcwd()



    for i, case in enumerate(start_cases):
        # Check if the output folder exists and delete it if it does
        case_folder = f'case_{case}'
        if os.path.exists(case_folder):
            continue
        # Create the output folder
        os.makedirs(case_folder, exist_ok=True)
        # Copy meshes 
        shutil.copy(f'{mesh_folder}/bar-1.g', case_folder)
        shutil.copy(f'{mesh_folder}/bar-2.g', case_folder)

        bar_1 = copy.copy(bar_1_template)
        bar_1['time integrator'] = integrator_templates[case_map[i][0]]
        bar_1['solver'] = solver_templates[case_map[i][0]]
        bar_1['boundary conditions']['Schwarz contact'][0]['swap BC types'] = case_map[i][2]

        with open(f'{case_folder}/bar-1.yaml', 'w',  encoding='utf-8') as file:
            yaml.dump(bar_1, file, default_flow_style=False, allow_unicode=True)

        bar_2 = copy.copy(bar_2_template)
        bar_2['time integrator'] = integrator_templates[case_map[i][1]]
        bar_2['solver'] = solver_templates[case_map[i][1]]
        bar_2['boundary conditions']['Schwarz contact'][0]['swap BC types'] = case_map[i][2]

        with open(f'{case_folder}/bar-2.yaml', 'w', encoding='utf-8') as file:
            yaml.dump(bar_2, file, default_flow_style=False, allow_unicode=True)

        replace_placeholders_in_file(f'{case_folder}/bar-1.yaml')
        replace_placeholders_in_file(f'{case_folder}/bar-2.yaml')

        main_yaml = copy.copy(schwarz_main_template)
        main_yaml['time step'] = ts
        main_yaml['CSV output interval'] = csv_output_int
        main_yaml['Exodus output interval'] = csv_output_int*2
        
        with open(f'{case_folder}/bars.yaml', 'w', encoding='utf-8') as file:
            yaml.dump(main_yaml, file, default_flow_style=False, allow_unicode=True)

        os.chdir(case_folder)
        command = [
                'julia', 
                '--project=@.', '-t', '6',
                '/Users/bphung/Work/software/Norma.jl/src/Norma.jl',
                'bars.yaml'
            ]
        # Run the command
        success = None
        try:
            f = open(f'{case_folder}.log', 'w')
            subprocess.run(command, check=True, stdout=f)
            f.close()
            success = True
        except:
            success = False
        os.chdir(pwd)
        if success == False:
            case_folder_failure = str(case_folder) + '_failure'
            shutil.move(case_folder, case_folder_failure)
            #raise Exception
        
create_cases('mesh_0', 0, main_yaml_ts[0], 32)
create_cases('mesh_1', 4, main_yaml_ts[1], 32)
create_cases('mesh_2', 8, main_yaml_ts[2], 32)
create_cases('mesh_3', 12, main_yaml_ts[3], 32)

Replacements made in 'case_0/bar-1.yaml' successfully.
Replacements made in 'case_0/bar-2.yaml' successfully.
Replacements made in 'case_3/bar-1.yaml' successfully.
Replacements made in 'case_3/bar-2.yaml' successfully.
Replacements made in 'case_6/bar-1.yaml' successfully.
Replacements made in 'case_6/bar-2.yaml' successfully.
Replacements made in 'case_7/bar-1.yaml' successfully.
Replacements made in 'case_7/bar-2.yaml' successfully.
Replacements made in 'case_10/bar-1.yaml' successfully.
Replacements made in 'case_10/bar-2.yaml' successfully.
Replacements made in 'case_11/bar-1.yaml' successfully.
Replacements made in 'case_11/bar-2.yaml' successfully.
Replacements made in 'case_12/bar-1.yaml' successfully.
Replacements made in 'case_12/bar-2.yaml' successfully.
